In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS my_catalog.bronze;
CREATE SCHEMA IF NOT EXISTS my_catalog.silver;
CREATE SCHEMA IF NOT EXISTS my_catalog.gold;

In [0]:
url = "jdbc:sqlserver://basil-jarvis.database.windows.net:1433;database=free-sql-db-123456;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

transactions_df = (spark.read
    .format("jdbc")
    .option("url", url)
    .option("dbtable", "transactions_data")
    .option("user", "CloudSAe5b9c2cc")
    .option("password", "")
    .load()
)

display(transactions_df.limit(5))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
9807093,2011-07-16 21:28:00,1169,4778,$13.53,Swipe Transaction,10782,Round Rock,TX,78665.0,5813,null
9807094,2011-07-16 21:28:00,1379,5215,$25.55,Swipe Transaction,54440,Clinton Township,MI,48036.0,5912,null
9807095,2011-07-16 21:29:00,194,5062,$34.90,Swipe Transaction,59935,Gallup,NM,87305.0,5499,null
9807096,2011-07-16 21:29:00,1407,5384,$0.94,Swipe Transaction,84425,Bessemer,AL,35023.0,5812,null
9807099,2011-07-16 21:31:00,1024,1006,$15.52,Swipe Transaction,41904,Andover,KS,67002.0,5813,null


In [0]:
cards_df = spark.table("jarvis_etl.default.cards_data")
display(cards_df.limit(5))

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,393314135668401,04/2024,866,YES,2,$33900,01/1991,2014,No
1,550,Mastercard,Credit,5278231764792292,06/2024,396,YES,1,$11600,01/1994,2013,No
2,556,Mastercard,Debit,5889825928297675,09/2021,422,YES,1,$19948,01/1995,2011,No
3,1937,Visa,Credit,4289888672554714,04/2020,736,YES,2,$16400,01/1995,2015,No
4,1981,Mastercard,Debit,5433366978583845,03/2024,530,YES,2,$19439,01/1997,2007,No


In [0]:
users_df = spark.read.format("csv").option("header","true").load("abfss://jarvis-bronze-container@jarvisstorage12345.dfs.core.windows.net/users_data.csv")

display(users_df.limit(5))

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [0]:
transactions_df.write.mode("overwrite").format("delta").saveAsTable("my_catalog.bronze.transactions_data")
cards_df.write.mode("overwrite").format("delta").saveAsTable("my_catalog.bronze.cards_data")
users_df.write.mode("overwrite").format("delta").saveAsTable("my_catalog.bronze.users_data")


In [0]:
%sql
SHOW TABLES IN my_catalog.bronze;

database,tableName,isTemporary
bronze,cards_data,false
bronze,transactions_data,false
bronze,users_data,false
